In [0]:
%sql

CREATE OR REPLACE TABLE dev.sales.customers_masking_drill (
    customer_id INT,
    full_name STRING,
    email STRING,
    phone STRING,
    last_4_ssn STRING,
    segment STRING
)
USING DELTA;

INSERT INTO dev.sales.customers_masking_drill (
    customer_id,
    full_name,
    email,
    phone,
    last_4_ssn,
    segment
)
VALUES
    (1, 'Anna Kowalska', 'anna.kowalska@example.com', '+48 501 100 200', '1234', 'premium'),
    (2, 'Jan Nowak', 'jan.nowak@example.com', '+48 502 200 300', '5678', 'standard'),
    (3, 'Maria Zielinska', 'maria.zielinska@example.com', '+48 503 300 400', '9012', 'premium'),
    (4, 'Piotr Wisniewski', 'piotr.wisniewski@example.com', '+48 504 400 500', '3456', 'standard'),
    (5, 'Katarzyna Wójcik', 'katarzyna.wojcik@example.com', '+48 505 500 600', '7890', 'vip');

In [0]:
%sql
SELECT *
FROM dev.sales.customers_masking_drill;

In [0]:
%sql
SELECT
  current_user() AS current_user,
  is_account_group_member('data_admins') AS is_data_admin,
  is_account_group_member('analysts') AS is_analyst,
  is_account_group_member('engineering') AS is_engineering;


In [0]:
%sql

CREATE OR REPLACE FUNCTION dev.sales.mask_email_multi(email STRING)
RETURNS STRING
RETURN CASE
  WHEN is_account_group_member('data_admins') THEN email
  WHEN is_account_group_member('analysts') THEN regexp_replace(email, '(?<=.{2}).(?=[^@]*?@)', '*')
  ELSE 'REDACTED'
END;

In [0]:
%sql

CREATE OR REPLACE FUNCTION dev.sales.mask_phone_multi(phone STRING)
RETURNS STRING
RETURN CASE
  WHEN is_account_group_member('data_admins') THEN phone
  WHEN is_account_group_member('analysts') THEN concat('*** *** ', right(regexp_replace(phone, '[^0-9]', ''), 4))
  ELSE 'REDACTED'
END;

In [0]:
%sql

CREATE OR REPLACE FUNCTION dev.sales.mask_last4_ssn_multi(last_4_ssn STRING)
RETURNS STRING
RETURN CASE
  WHEN is_account_group_member('data_admins') THEN last_4_ssn
  ELSE '****'
END;

In [0]:
%sql

ALTER TABLE dev.sales.customers_masking_drill
ALTER COLUMN email SET MASK dev.sales.mask_email_multi;

ALTER TABLE dev.sales.customers_masking_drill
ALTER COLUMN phone SET MASK dev.sales.mask_phone_multi;

ALTER TABLE dev.sales.customers_masking_drill
ALTER COLUMN last_4_ssn SET MASK dev.sales.mask_last4_ssn_multi;

In [0]:
%sql
SELECT *
FROM dev.sales.customers_masking_drill;